[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarondomenzain/tracking-softmatter-aarond/blob/tracking-softmatter-aarond-dev/tutorial/tracking/tracking_spheres.ipynb)

# Particle Tracking Tutorial: Trajectory Linking

In this tutorial, you’ll explore different methods to link particle localizations across time to recostruct trajectories using both simulated data and real experimental images.

You’ll start by generating a simulated movie of microscopic particles undergoing Brownian motion, mimicking what you might see in a soft matter or biophysics experiment. For each frame, you'll use a U-net, a supervised neural network, to detect and localize particles. Then comes the core challenge: linking localization into trajectories.

Here you’ll test and compare various methods to perform the linking of the trajectories:

- Nearest-neighbor linking (using TrackPy — a classic in particle tracking)

- Linear Assignment Problem (LAP) (using LapTrack - a more flexible and general framework)

- MAGIK (a geometric deep learning method based on graph neural networks)

You’ll be using Python libraries like NumPy, SciPy, Matplotlib, scikit-image, PyTorch, DeepTrack, and Deeplay. 

## Table of Contents

0. [Importing the Required Libraries and Loading Utility Functions](#importing-the-required-libraries-and-loading-utility-functions)
1. [Loading and Visualizing Experimental Videos](#loading-and-visualizing-experimental-videos)
2. [Simulating Realistic Videos with DeepTrack](#simulating-realistic-videos-with-deeptrack)
    - [Simulating a Single Particle](#simulating-a-single-particle)
    - [Simulating a Video Frame](#simulating-a-video-frame)
    - [Simulating Brownian Trajectories](#simulating-brownian-trajectories)
    - [Simulating a Video](#simulating-a-video)

2. [Detecting and Localizing Particles with U-net](#detecting-and-localizing-particles-with-u-net)
    - [Training U-net with Experiments](#training-u-net-with-experiments)
    - [Evaluating U-net on Simulations](#evaluating-u-net-on-simulations)
    - [Applying U-net to Simulations](#applying-u-netmulations)
    - [Applying U-net to Experiments](#applying-u-net-to-experiments)

3. [Method 1: Nearest-neighbor Linking with TrackPy](#method-1-nearest-neighbor-linking-with-trackpy)
    - [Linking Localizations in Simulations](#linking-localizations-in-simulations)
    - [Evaluating Linking Performance](#evaluating-linking-performance)
    - [Linking Localizations in Experiments](#linking-localizations-in-experiments)

4. [Method 2: Linear Assignment Problem (LAP) with LapTrack](#method-2-linear-assignment-problem-lap-with-laptrack)
    - [Linking Localizations in Simulations](#linking-localizations-in-simulations)
    - [Evaluating Linking Performance](#evaluating-linking-performance)
    - [Linking Localizations in Experiments](#linking-localizations-in-experiments)

5. [Method 3: MAGIK](#method-3-magik)
    - [Training MAGIK with Simulations](#training-magik-with-simulations)
    - [Linking Localizations in Simulations](#linking-localizations-in-simulations)
    - [Evaluating Linking Performance](#evaluating-linking-performance)
    - [Linking Localizations in Experiments](#linking-localizations-in-experiments)



## Importing the Required Libraries and Loading Utility Functions

Uncomment the next cell if running on Google Colab/Kaggle.

In [ ]:
#!pip install deeptrack deeplay trackpy laptrack -q

In [ ]:
# Standard libraries.
import logging
import os
import random
import sys

# Configuration
import matplotlib
matplotlib.rcParams["animation.embed_limit"] = 60  # Larger animations inline
logging.disable(logging.WARNING)  # Suppress warnings and below

# Core Scientific Stack
import numpy as np
import pandas as pd

# Plotting and Display
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Machine Learning
import deeplay as dl
from sklearn.metrics import f1_score
import torch
from torchvision.transforms import Compose
from torch_geometric.loader import DataLoader

# Particle Tracking and Simulation
import deeptrack as dt
from laptrack import LapTrack
import trackpy as tp

Load a set of custom functions defined specifically for this notebook from the `utils` directory. For detailed documentation of each function, refer to the comments and docstrings within the files.

In [ ]:
# Load functions and utilities for dataset generation and visualization.
# Sys append a folder to the path.
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import all the functions contained in the folder utils.
import utils

Set random seeds to make results reproducible across runs, especially during training and data simulation. Also select the best device for computations with Torch.

In [ ]:
# Set a fixed seed value.
seed = 98

# Python, NumPy, and PyTorch (CPU).
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Only set CUDA seeds if a GPU is available.
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Seeds set to {seed} (with CUDA: {torch.cuda.is_available()})")

# Get the best available device for Torch computation.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Using CPU")

## Loading and Visualizing Experimental Videos

You'll use experimental video of a system of colloidal particles recorded with fluorescence microscopy. Data from https://www.nature.com/articles/s41467-022-30497-z.


In [ ]:
# Define the folder and video file name.
video_folder = "videos"
video_file_name = "experimental_video.npy"

# Construct the full path.
video_path = os.path.join(video_folder, video_file_name)

# Load the video data.
exp_video = np.load(video_path)

utils.play_video(exp_video, "Experimental Video")

Display the first frame of the video; then manually select and display a single particle by specifying its centroid coordinates (x, y) and a box width.

In [ ]:
# Select the first frame of the video.
exp_image = exp_video[0]

# Get the shape of the image.
assert exp_image.shape[0] == exp_image.shape[1], "Warning: Image not square!"
exp_image_size = exp_image.shape[0]

Specify the parameters for the viewing box.

In [ ]:
# Box size to zoom in an individual particle.
exp_crop_size = 15

# Coordinates of the center of the particle.
x_center = 96
y_center = 41

In [ ]:
# # Calculate top-left corner of the crop.
x = x_center - exp_crop_size // 2
y = y_center - exp_crop_size // 2

# Select a crop as a subset of the entire image.
exp_crop = exp_image[
    y:y + exp_crop_size,  # row (y)
    x:x + exp_crop_size,  # column (x)
]

# Initialize figure instance.
fig = plt.figure()

vmin, vmax = np.percentile(exp_image, [1, 99])

# Draw a red rectangle around the crop.
fig.add_subplot(111)
plt.imshow(exp_image, cmap="gray", vmin=vmin, vmax=vmax)
plt.title("Experimental Image", size=13)
plt.plot([x, x, x + exp_crop_size, x + exp_crop_size, x],
         [y, y + exp_crop_size, y + exp_crop_size, y, y], 'r-')
plt.axis("off")

# Plot the rectangle on the top right corner.
fig.add_subplot(555)
plt.imshow(exp_crop, cmap="gray", vmin=vmin, vmax=vmax)
plt.axis("off")
plt.show()

## Simulating Realistic Videos with DeepTrack

DeepTrack allows you to simulate physically realistic microscopy images and videos, enabling precise control over imaging parameters and particle properties. These simulations provide ground-truth data, making them ideal for benchmarking classical and AI-based tracking methods, as well as for training neural networks in a controlled environment.

### Simulating a Single Particle

Adjust the simulation parameters to accurately replicate the features observed in the cropped region of the experimental image.

In [ ]:
# Same as the box width.
sim_crop_size = exp_crop_size  

# Size of a pixel in nanometers in the output image.
pixel_size_nm = 640 # In nm.

# Radius of the particle.
particle_radius = 440 # In nm.

# Define central spherical scatterer.
sphere = dt.Sphere(
    position=0.5 * np.array([sim_crop_size, sim_crop_size]) + (-0.5, 0.5),
    z= 400 * dt.units.nm, # Particle in focus.
    radius= particle_radius * dt.units.nm,  # Radius in nm
    intensity= 1E5,  # Field magnitude squared
    refractive_index=1.59,
)

# Simulate the properties of the fluorescence microscope.
optics = dt.Fluorescence(
    NA=0.3,  # Numerical aperture
    wavelength=508 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_crop_size, sim_crop_size],
    magnification=2.6,
    resolution=pixel_size_nm * dt.units.nm,  # Camera effective resolution
)

# Apply transformations.
sim_crop = (
    optics(sphere)
    >> dt.Background(750)  # Background intensity level
    >> dt.Poisson(snr=6000)  # Signal-to-noise ratio (SNR) of the image
)

# Turn the crop into a NumPy array.
sim_crop = np.squeeze(sim_crop())

# Plot the simulated and experimental crops.
fig, axes = plt.subplots(1, 2)

# Simulated crop.
plot = axes[0].imshow(sim_crop, cmap="gray")
axes[0].axis("off")
axes[0].set_title("Simulated Crop")

# Experimental crop.
axes[1].imshow(exp_crop, cmap="gray")  
axes[1].axis("off")
axes[1].set_title("Experimental Crop")

# Adjust layout and show plot.
plt.tight_layout()
plt.show()

Extract and visualize the raw intensity profiles along the central horizontal and vertical lines of both the simulated and experimental crops. This comparison helps evaluate how well the simulation reproduces the intensity distribution observed in real microscopy images.

In [ ]:
# Compute the center index.
center = sim_crop_size // 2

# Extract horizontal (row) profiles.
sim_horiz = sim_crop[center, :]
exp_horiz = exp_crop[center, :]

# Extract vertical (column) profiles.
sim_vert = sim_crop[:, center]
exp_vert = exp_crop[:, center]

# Create a 1×2 subplot.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), tight_layout=True,sharey=True)

# --- Horizontal profile ---
axes[0].plot(sim_horiz, label="Simulated Crop", color="orange")
axes[0].plot(exp_horiz, label="Experimental Crop", color="blue")
axes[0].set_xlabel("Pixel (x)")
axes[0].set_ylabel("Intensity")
axes[0].set_title("Horizontal Intensity Profile (Center Row)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# --- Vertical profile ---
axes[1].plot(sim_vert, label="Simulated Crop", color="orange")
axes[1].plot(exp_vert, label="Experimental Crop", color="blue")
axes[1].set_xlabel("Pixel (y)")
axes[1].set_ylabel("Intensity")
axes[1].set_title("Vertical Intensity Profile (Center Column)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

### Simulating a Video Frame

Create a simulated image containing non-overlapping spherical particles. Begin by generating their coordinates to serve as the ground-truth positions. Then, use DeepTrack to render optically realistic particles at these coordinates, resulting in a physically plausible microscopy image.

In [ ]:
# Parameters of the simulation.
sim_image_size = 256
N_particles = 40
particle_radius = 440  # Particle radius in nm

# Dictionary for particle properties. Dimensions are set with lambda
# functions to introduce variety to the dataset.
sphere_properties = {
    "intensity": lambda: np.random.uniform(1.0, 2.0) * 0.85E5,
    "z": lambda: np.random.uniform(-1000, -20000) * dt.units.nm,
    "radius": particle_radius * dt.units.nm,
    "refractive_index": 1.59,
}
# Set the optical properties of the microscope.  This dictionary is a DeepTrack
# optics object.
optics_properties = dt.Fluorescence(
    NA=0.4,  # Numerical aperture
    wavelength=508 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, sim_image_size, sim_image_size],
    magnification=2.5,
    resolution=640 * dt.units.nm,  # Camera effective resolution
)
# Generate ground truth positions. 
sim_gt_pos = utils.generate_centroids(
    num_particles=N_particles,
    fov_size=sim_image_size,
    particle_radius=particle_radius,
)
# Simulate image.
sim_image = utils.transform_to_video(
    sim_gt_pos,
    fov_size=sim_image_size,
    core_particle_props=sphere_properties,
    optics_props=optics_properties,
    background_props={"poisson_snr":6500, "background_mean": 650},
)

Visualize the simulated image and compare it with the experimental one.

In [ ]:
# Plot the simulated and experimental images.
fig, axes = plt.subplots(1, 2)
vmin, vmax = np.percentile(exp_image, [1, 99])

# Simulated image.
axes[0].imshow(sim_image, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].axis("off")
axes[0].set_title("Simulated Image")
axes[0].scatter(sim_gt_pos[:, 1], sim_gt_pos[:, 0], marker=".", color="red", s=10, label="Ground truth positions")
axes[0].legend(loc="upper left", markerscale=5)

# Experimental image.
axes[1].imshow(exp_image, cmap="gray", vmin=vmin, vmax=vmax)  
axes[1].axis("off")
axes[1].set_title("Experimental Image")

# Adjust layout and show plot.
plt.tight_layout()
plt.show()

Perform a more quantitative comparison by plotting the intensity histograms.

In [ ]:
# Flatten the arrays to 1D.
sim_vals = sim_image.ravel()
exp_vals = exp_image.ravel()

# Compute common bin edges.
all_vals = np.concatenate([sim_vals, exp_vals])
num_bins = 60
bins = np.linspace(all_vals.min(), np.quantile(all_vals, 0.99), num_bins + 1)

# Create figure with two subplots sharing axes.
fig, axes = plt.subplots(
    1, 2, 
    figsize=(10, 4), 
    sharey=True, 
    sharex=True, 
    tight_layout=True
)

# Histogram for simulated image.
axes[0].hist(sim_vals, bins=bins, alpha=0.7, edgecolor="black")
axes[0].set_title("Simulated Image Histogram")
axes[0].set_xlabel("Intensity")
axes[0].set_ylabel("Pixel Count")

# Histogram for experimental image.
axes[1].hist(exp_vals, bins=bins, alpha=0.7, edgecolor="black")
axes[1].set_title("Experimental Image Histogram")
axes[1].set_xlabel("Intensity")

plt.show()

### Simulating Brownian Trajectories

You'll simulate a set of trajectories that visually resemble the experimental data to evaluate the performance of different tracking methods. The goal is to replicate the Brownian motion of nanoparticles as observed in the experimental videos. This is done using the `simulate_Brownian_trajs()` function from the utility file, which generates 2D trajectories based on a random walk model. Refer to the function for more details.

In [ ]:
# Simulation parameters.
number_particles = 30
number_timesteps = 50

# Simulate trajectories for one video.
sim_trajs_gt = utils.simulate_Brownian_trajs(
    num_particles=number_particles,
    num_timesteps=number_timesteps,
    fov_size=sim_image_size,
    diffusion_std=0.5,  # Corresponds to sqrt(2Dt)
)

For evaluation purposes, trajectories that move out and back in the field of view due to boundary conditions are treated as separate trajectories. For further analysis, the trajectories are transformed into a list.

In [ ]:
# Break trajectories going in/out of FOV.
sim_trajs_gt_list = utils.traj_break(
    trajs=sim_trajs_gt,
    fov_size=sim_image_size,
    num_particles=sim_trajs_gt.shape[1],
)

### Simulating a Video

You'll now simulate a video of particle motion that resembles experimental data and compares the two by playing them simultaneously.

In [ ]:
# Simulate video.
sim_video = utils.transform_to_video(
    np.delete(sim_trajs_gt, 2, 2),  # Remove frame axis
    fov_size=sim_image_size,
    core_particle_props=sphere_properties,
    optics_props=optics_properties,
    background_props={"poisson_snr": 9500, "background_mean": 650},
    save_video=True,
    path="videos/simulated_video.tiff",
)

# Play both simulated and experimental videos and compare.
utils.play_video(sim_video, "Simulated Video")
utils.play_video(exp_video, "Experimental Video")

**Note:** Several out-of-focus particles are rather dim and not clearly visible unless the dynamic range of the video is adjusted. However, including them is essential to accurately reproduce the experimental conditions, especially in terms of intensity distribution and background noise characteristics.

## Detecting and Localizing Particles with U-net

A U-net will be used to detect the position of particles at each frame of the video, similarly as shown in the **Detections** notebooks of this tutorial. These positions will be passed to different linking methods to build trajectories and compare their performance.

### Training a U-net with Simulated Data

Implement a simulation pipeline, as shown in the Detection notebooks, to generate a training dataset for a U-Net.

In [ ]:
# Number of samples, image size, and particles.
num_samples = 256
train_image_size = 128
max_num_particles = 10
force_simulation = True  # Flag to force simulation even if data exists

# Optical properties of spheres with variability.
sphere_properties = {
    "intensity": lambda: np.random.uniform(1.0, 2.0) * 0.85E5, 
    "z": lambda: np.random.uniform(-4500, -25000) * dt.units.nm,
    "radius": particle_radius * dt.units.nm,
    "refractive_index": 1.59,
}

# Set the optical properties of the microscope.
optics_properties = dt.Fluorescence(
    NA=lambda: np.random.uniform(0.4, 0.6),  # Numerical aperture with variability
    wavelength=508 * dt.units.nm,
    refractive_index_medium=1.33,
    output_region=[0, 0, train_image_size, train_image_size],
    magnification= lambda: np.random.uniform(2.0, 3.5),
    resolution=640 * dt.units.nm,  # Camera effective resolution
)

# Create path to store training dataset.
folder_name = "UNet"
training_dataset_filename = "UNet_training_dataset_spheres.npz"
training_dataset_folder = os.path.join(folder_name, "training_data")
training_dataset_filepath = os.path.join(
    training_dataset_folder, 
    training_dataset_filename,
)

# Create the enclosing directory if not existent already.
if not os.path.exists(training_dataset_folder):
    os.makedirs(training_dataset_folder, exist_ok=True)

# Try to load preexisting data, if not available or forced, raise an exception
# error to generate new data.
try:
    if force_simulation: 
        # Raise the exception error if simulation is forced.
        raise FileNotFoundError("Forced simulation by user request.")
    
    if not os.path.isfile(training_dataset_filepath):
        # If file is not found, start training.    
        raise FileNotFoundError(
            "Training dataset file not found. Starting simulation."
        )
    
    # Load existing data
    data = np.load(training_dataset_filepath)
    images = data["images"]
    print(images.shape)
    maps = data["maps"]
    Nsamples = len(images)
    print(f"Loaded file: {training_dataset_filepath}")
        
# Handle the case of either file not found or forced training.
except FileNotFoundError:
    # Generate new dataset if file not found or simulation is forced.
    images, maps = utils.generate_particle_dataset(
        num_samples = num_samples,
        fov_size = train_image_size,
        max_num_particles = max_num_particles,
        core_particle_dict=sphere_properties,
        optics_properties=optics_properties,
        background_props={"poisson_snr": 6500, "background_mean": 2500},
    )
    
    # Save the simulated training dataset.
    np.savez(training_dataset_filepath, images=images, maps=maps)
    print(f"Training dataset saved in: {training_dataset_filepath}.")

In [ ]:
# Select an image and its corresponding probability maps and mask to show.
selected_image_index = np.random.randint(0, len(images))

# Extract the image and probability map from 4D arrays.
selected_image = np.squeeze(images[selected_image_index])
selected_probability_map = np.squeeze(maps[selected_image_index])

# Plot the image as the first subplot.
utils.plot_image_mask_ground_truth_map(
    image=selected_image,
    gt_map=selected_probability_map,
    title=f"Training dataset element: {selected_image_index+1}/{len(images)}"
)

Create a U-net model and a regressor.

In [ ]:
unet = dl.UNet2d(
    in_channels=1, 
    channels=[32, 64, 128, 256, 512], 
    out_channels=1,
)
regressor_unet = dl.Regressor(
    model=unet, 
    loss=torch.nn.MSELoss(), 
    optimizer=dl.AdamW(),
).create()

# Image selector with a random picker. This is performed in order to properly 
# link an element in maps array with its corresponding element in images 
# array.
selector = dt.Lambda(
    lambda i: lambda x: x[i], i=lambda l: np.random.randint(l), l=len(images)
)

# Apply augmentations of added Gaussian noise only to images.
images_augmentation_pipeline = (
    dt.Value(images)
    >> dt.NormalizeMinMax(0.0, 1.0)
    >> dt.Gaussian(0, 0.002)
    >> dt.Poisson(snr=50)
    >> dt.NormalizeMinMax(
        lambda: np.random.uniform(0.0, 0.1), 
        lambda: np.random.uniform(0.9, 1.0),
    )
)

maps_pipeline = dt.Value(maps) >> dt.NormalizeMinMax(0.0, 1.0)

pipeline = (
    (images_augmentation_pipeline & maps_pipeline)
    >> selector
    # >> dt.FlipUD()
    # >> dt.FlipLR()
    >> dt.MoveAxis(-1, 0)
    >> dt.pytorch.ToTensor(dtype=torch.float)
)
# Sanity check.
sanity_check_pipeline_augmentation = np.squeeze(pipeline.update().resolve())
sanity_check_image_augmentation = sanity_check_pipeline_augmentation[0]
sanity_check_map_augmentation = sanity_check_pipeline_augmentation[1]

# Plot the image as the first subplot.
utils.plot_image_mask_ground_truth_map(
    image=sanity_check_image_augmentation,
    gt_map=sanity_check_map_augmentation,
    title=f"Random Augmentation from Training Pipeline",
)

In [ ]:
train_dataset = dt.pytorch.Dataset(pipeline, length=256)

train_loader = DataLoader(
    train_dataset, 
    batch_size=8, 
    shuffle=True,
)

trainer_unet = dl.Trainer(max_epochs=200, accelerator="auto")

Check whether pre-trained U-net weights already exist on disk. If they are missing or if training is explicitly forced, the model is trained on the experimental crops using the specified training pipeline, and the resulting weights are saved for future use.

If the weights are already available and training is not forced, they are simply loaded from file.

Afterward, the model is set to evaluation mode, which disables training-specific behaviors, ensuring consistent behavior during inference.

In [ ]:
# Force training if desired.
force_training = True

# Define the file paths for the model weights.
unet_path = "UNet_model_spheres.pth"
regressor_unet_path = "UNet_reg_spheres.pth"

# Define folder and construct full file paths.
folder_name = "UNet"
unet_path = os.path.join(folder_name, unet_path)
regressor_unet_path = os.path.join(folder_name, regressor_unet_path)

# Load preexisting weights if they exist and training is not forced.
if (not force_training and os.path.exists(unet_path)
    and os.path.exists(regressor_unet_path)):
    unet.load_state_dict(torch.load(unet_path, weights_only=True))
    regressor_unet.load_state_dict(
        torch.load(regressor_unet_path, weights_only=True)
    )
    print(f"Loaded preexisting U-Net weights from '{folder_name}/'.")
else:
    print("Training U-Net model (either forced or weights not found).")
    
    # Ensure the save directory exists.
    os.makedirs(folder_name, exist_ok=True)

    # Train the U-Net model.
    trainer_unet.fit(regressor_unet, train_loader)

    # Monitor training history.
    trainer_unet.history.plot()
    
    # Save the weights.
    torch.save(unet.state_dict(), unet_path)
    torch.save(regressor_unet.state_dict(), regressor_unet_path)
    print(f"Saved trained U-Net weights to '{folder_name}/'.")
    
# Transfer the model to the best available device (optional).
regressor_unet.to(device);

### Evaluating U-net on Simulated Data

Apply the trained U-net model to a frame of the simulated video.  Set the inference parameters that control detection sensitivity. Get the prediction features, which can be useful for visualizing detections. Extract the final coordinates of the detected particles and print how many particles were detected. Plot the predicted localizations from U-Net alongside the ground truth on the simulated image and quantify the performance.

In [ ]:
# Extract intensity corresponding to first and 99th percentile of intensity distribution.
p1, p99 = np.percentile(sim_video[0], [1, 99])

# Apply contrast stretching to enhance contrast.
test_frame = np.clip(sim_video[0], p1, p99)

# Normalize intensity to [0,1] for inference with U-Net.
test_frame = utils.normalize_min_max(test_frame)

# Convert the image to analyze into a PyTorch tensor.
formatted_sim_image = utils.format_image(test_frame)

# Transfer the tensor to the best available device (optional).
formatted_sim_image = formatted_sim_image.to(device)

# Apply the UNet to the loaded image.
sim_image_pred_map_tensor = regressor_unet(formatted_sim_image.to(device))

# Convert to NumPy array.
sim_image_pred_map = sim_image_pred_map_tensor[0, 0, :, :].cpu().detach().numpy()

# Normalize the predicted ground truth map for thresholding purposes.
sim_image_pred_map = utils.normalize_min_max(sim_image_pred_map, minimum_value=0.0, maximum_value=1.0)

# Apply contrast stretching to the predicted ground truth map for automatic thresholding.
p1, p98 = np.percentile(sim_image_pred_map, [1, 98])

# Apply thresholding to the predicted ground truth map.
sim_image_pred_mask_unet = sim_image_pred_map > p98

# Convert the masked ground truth map to positions.
sim_locs_pred_method = \
    utils.mask_to_positions(sim_image_pred_mask_unet, sim_image_pred_map)

# Plot the simulated image with the positions predicted by U-Net.
utils.plot_predicted_positions(
    image=test_frame, 
    pred_positions=sim_locs_pred_method, 
    gt_positions=sim_trajs_gt[0][:,:2],
    title="Method 3 - Simulated Image",
)

# Plot the predicted ground truth map and its masked version.
utils.plot_image_mask_ground_truth_map(
    mask=sim_image_pred_mask_unet,
    gt_map=sim_image_pred_map,
    title="Method 3 - Simulated Image",
)

# Print the number of detections.
print(f"Found {len(sim_locs_pred_method[:,1])} detections.")

### Evaluating U-net on Experimental Data

Apply the trained U-net model to a frame of the experimental video.  Set the inference parameters that control detection sensitivity. Get the prediction features, which can be useful for visualizing detections. Extract the final coordinates of the detected particles and print how many particles were detected. Plot the predicted localizations from U-Net alongside the ground truth on the simulated image and quantify the performance.

In [ ]:
# Extract intensity corresponding to first and 98th percentile of intensity distribution.
p1, p98 = np.percentile(exp_video[0], [1, 98])

# Apply contrast stretching to enhance contrast.
test_frame = np.clip(exp_video[0], p1, p98)

# Normalize intensity to [0,1] for inference with U-Net.
test_frame = utils.normalize_min_max(test_frame)

# Convert the image to analyze into a PyTorch tensor.
formatted_exp_image = utils.format_image(test_frame)

# Transfer the tensor to the best available device (optional).
formatted_exp_image = formatted_exp_image.to(device)

# Apply the UNet to the loaded image.
exp_image_pred_map_tensor = regressor_unet(formatted_exp_image.to(device))

# Convert to NumPy array.
exp_image_pred_map = exp_image_pred_map_tensor[0, 0, :, :].cpu().detach().numpy()

# Normalize the predicted ground truth map for thresholding purposes.
exp_image_pred_map = utils.normalize_min_max(exp_image_pred_map, minimum_value=0.0, maximum_value=1.0)

# Apply contrast stretching to the predicted ground truth map for automatic thresholding.
p1, p98 = np.percentile(exp_image_pred_map, [1, 98])

# Apply thresholding to the predicted ground truth map.
exp_image_pred_mask_unet = exp_image_pred_map > p98

# Convert the masked ground truth map to positions.
exp_locs_pred_method = \
    utils.mask_to_positions(exp_image_pred_mask_unet, exp_image_pred_map)

# Plot the experimental image with the positions predicted by U-Net.
utils.plot_predicted_positions(
    image=test_frame, 
    pred_positions=exp_locs_pred_method, 
    title="Method 3 - Experimental Image",
)

# Plot the predicted ground truth map and its masked version.
utils.plot_image_mask_ground_truth_map(
    mask=exp_image_pred_mask_unet,
    gt_map=exp_image_pred_map,
    title="Method 3 - Experimental Image",
)

# Print the number of detections.
print(f"Found {len(exp_locs_pred_method[:,1])} detections.")

### Applying U-Net to a Simulated Video
Iteratively apply U-Net to every frame of the simulated video and store localizations in a dataframe.

In [ ]:
df_sim_video = []
for frame_index, sim_frame in enumerate(sim_video):

    # Extract intensity corresponding to first and 99th percentile of intensity distribution.
    p1, p99 = np.percentile(sim_frame, [1, 99])

    # Apply contrast stretching to enhance contrast.
    sim_frame = np.clip(sim_frame, p1, p99)

    # Normalize intensity to [0,1] for inference with U-Net.
    sim_frame = utils.normalize_min_max(sim_frame, minimum_value=0.0, maximum_value=1.0)

    # Convert the image to analyze into a PyTorch tensor.
    sim_frame_formatted = utils.format_image(sim_frame)

    # Transfer the tensor to the best available device (optional).
    sim_frame_formatted = sim_frame_formatted.to(device)

    # Apply the UNet to the loaded image.
    sim_image_pred_map_tensor = regressor_unet(sim_frame_formatted)

    # Convert to NumPy array.
    sim_image_pred_map = sim_image_pred_map_tensor[0, 0, :, :].cpu().detach().numpy()

    # Normalize the predicted ground truth map for thresholding purposes.
    sim_image_pred_map = utils.normalize_min_max(sim_image_pred_map, minimum_value=0.0, maximum_value=1.0)

    # Apply contrast stretching to the predicted ground truth map for automatic thresholding.
    p1, p98 = np.percentile(sim_image_pred_map, [1, 98])
    
    # Apply thresholding to the predicted ground truth map.
    sim_image_pred_mask_unet = sim_image_pred_map >  p98

    # Convert the masked ground truth map to positions.
    sim_locs_pred = \
        utils.mask_to_positions(sim_image_pred_mask_unet, sim_image_pred_map)

    # Store detections in a DataFrame.
    df_frame = pd.DataFrame(sim_locs_pred, columns=["x", "y"])
    df_frame["frame"] = frame_index
    df_sim_video.append(df_frame)

    # Print no. of detections every 10 frames.
    if frame_index % 10 == 0:
        print(f"Detections in frame {frame_index}: {len(sim_locs_pred)}")

# Combine all detections into a single DataFrame.
df_sim_video = pd.concat(df_sim_video, ignore_index=True)

### Applying U-Net to an Experimental Video
Iteratively apply U-Net to every frame of the experimental video and store localizations in a dataframe.

In [ ]:
df_exp_video = []
for frame_index, exp_frame in enumerate(exp_video):

    # Extract intensity corresponding to first and 98th percentile of intensity distribution.
    p1, p98 = np.percentile(exp_frame, [1, 99])

    # Apply contrast stretching to enhance contrast.
    exp_frame = np.clip(exp_frame, p1, p98)

    # Normalize intensity to [0,1] for inference with U-Net.
    exp_frame = utils.normalize_min_max(exp_frame, minimum_value=0.0, maximum_value=1.0)

    # Convert the image to analyze into a PyTorch tensor.
    formatted_exp_image = utils.format_image(exp_frame)

    # Transfer the tensor to the best available device (optional).
    formatted_exp_image = formatted_exp_image.to(device)

    # Apply the UNet to the loaded image.
    exp_image_pred_map_tensor = regressor_unet(formatted_exp_image)

    # Convert to NumPy array.
    exp_image_pred_map = exp_image_pred_map_tensor[0, 0, :, :].cpu().detach().numpy()

    # Normalize the predicted ground truth map for thresholding purposes.
    exp_image_pred_map = utils.normalize_min_max(exp_image_pred_map, minimum_value=0.0, maximum_value=1.0)

    # Apply contrast stretching to the predicted ground truth map for automatic thresholding.
    p1, p98 = np.percentile(exp_image_pred_map, [1, 98])

    # Apply thresholding to the predicted ground truth map.
    exp_image_pred_mask_unet = exp_image_pred_map > p98

    # Convert the masked ground truth map to positions.
    exp_locs_pred = \
        utils.mask_to_positions(exp_image_pred_mask_unet, exp_image_pred_map)

    # Store detections in a DataFrame.
    df_frame = pd.DataFrame(exp_locs_pred, columns=["x", "y"])
    df_frame["frame"] = frame_index
    df_exp_video.append(df_frame)

    # Print no. of detections every 10 frames.
    if frame_index % 10 == 0:
        print(f"Detections in frame {frame_index}: {len(exp_locs_pred)}")

# Combine all detections into a single DataFrame.
df_exp_video = pd.concat(df_exp_video, ignore_index=True)

## Method 1: Nearest-Neighbor Linking with TrackPy

TrackPy constructs trajectories by linking localized positions across frames using a predictive nearest-neighbor algorithm. The input must be a Pandas DataFrame containing the particle positions, usually with columns `x`, `y`, and `frame`.

See the [TrackPy tutorial on prediction and linking](https://soft-matter.github.io/trackpy/dev/tutorial/prediction.html) for more details on how the algorithm works and how to tune parameters like `search_range` and `memory`.

### Linking Localizations in Simulations

Apply the method to the localization dataframe.

In [ ]:
# Link detections across frames into trajectories using trackpy.link().
# The `search_range` parameter sets the maximum allowed displacement in pixels
# between frames, and the `memory` parameter allows particles to vanish for a
# given number of frames and still be linked to the same trajectory.
sim_trajs_pred_method1 = tp.link(
    df_sim_video,
    search_range=3,
    memory=4,
)

Create a trajectory list from the output dataframe.

In [ ]:
sim_trajs_pred_method1_list = []
for i in sim_trajs_pred_method1.particle.drop_duplicates():
    traj = sim_trajs_pred_method1.loc[
        sim_trajs_pred_method1.particle == i,
        ["frame", "x", "y"]
    ].values
    sim_trajs_pred_method1_list.append(traj)

print(f"Number of trajectories found: {len(sim_trajs_pred_method1_list)}")

Filter out trajectories shorter than 10 frames

In [ ]:
# Filter trajectory lists shorter than 10 frames.
sim_trajs_pred_method1_list = [trajectory for trajectory in sim_trajs_pred_method1_list if len(trajectory) >= 15]

print(f"Number of trajectories found: {len(sim_trajs_pred_method1_list)}")

Create a video with overlayed localizations and trajectories.

In [ ]:
sim_video_method1_results = utils.make_video_with_trajs(
    trajs_pred_list=sim_trajs_pred_method1_list,
    video=sim_video,
    fov_size=sim_image_size,
    trajs_gt_list=sim_trajs_gt_list,
    figure_title="Simulated video"
)

# Display the video.
sim_video_method1_results


### Evaluating Linking Performance

Evaluate the overall performance of the tracking (detection + linking) method using the following metrics:

- **TP (True Positives):** Number of ground-truth particles correctly matched to estimated positions.

- **FP (False Positives):** Number of estimated particles that do not correspond to any ground-truth particle.

- **FN (False Negatives):** Number of ground-truth particles that were not matched to any estimated position.

- **α:** A measure of the overall agreement between ground-truth and estimated tracks, ignoring unmatched (spurious) estimated tracks.

- **β:** A stricter version of α that penalizes unmatched (spurious) tracks, providing a more realistic performance score.  

See the detailed definitions in [Chenouard et al., Nature Methods, 2014](https://www.nature.com/articles/nmeth.2808).


In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_gt_list,
    sim_trajs_pred_method1_list,
    eps=5,
);

Display the reconstructed trajectories together with their groud truth.

In [ ]:
#  Compute the total squared distance between all trajectories to match
#  predicted trajectories with ground truth.
matched_pairs, _, _ = utils.trajectory_assignment(
    sim_trajs_gt_list,
    sim_trajs_pred_method1_list,
    eps=5,
)

# Plot the trajectories.
utils.plot_trajectory_matches(
    sim_trajs_gt_list, sim_trajs_pred_method1_list, matched_pairs,
)

Calculate the time-averaged MSD for all the trajectories and compare curves obtained for matching trajectories (same color).

In [ ]:
utils.plot_TAMSDs(
    trajs_pred=sim_trajs_pred_method1_list,
    trajs_gt=sim_trajs_gt_list,
    matched_pairs=matched_pairs,
) 

### Linking Localizations in Experiments

Apply the same steps to track the experiment and visualize the results.

In [ ]:
# Link detections across frames into trajectories using trackpy.link().
exp_trajs_pred_method1 = tp.link(
    df_exp_video,
    search_range=3, 
    memory=3,
)
# Create a list to store trajectories.
exp_trajs_pred_method1_list = []
for i in exp_trajs_pred_method1.particle.drop_duplicates():
    traj = exp_trajs_pred_method1.loc[
        exp_trajs_pred_method1.particle == i,
        ["frame", "x", "y"]
    ].values
    exp_trajs_pred_method1_list.append(traj)

# Filter trajectory lists shorter than 10 frames.
exp_trajs_pred_method1_list = [trajectory for trajectory in exp_trajs_pred_method1_list if len(trajectory) >= 15]
print(f"Number of trajectories found: {len(exp_trajs_pred_method1_list)}")

# Display the experimental video with the localizations and trajectories.
exp_video_method1_results = utils.make_video_with_trajs(
    trajs_pred_list=exp_trajs_pred_method1_list,
    video=exp_video,
    fov_size=exp_image_size,
)

# Display the video.
exp_video_method1_results

Calculate the time-averaged MSD for the trajectories.

In [ ]:
utils.plot_TAMSDs(trajs_pred=exp_trajs_pred_method1_list)

## Method 2: Linear Assignment Problem (LAP) with LapTrack


LapTrack solves the trajectory linking problem by formulating it as a Linear Assignment Problem (LAP), a well-established optimization approach in multi-object tracking. It builds a cost matrix that quantifies dissimilarity—typically based on spatial distance—between particle detections in consecutive frames. Lower distances correspond to lower costs and indicate higher likelihoods of correspondence.

LapTrack uses the Hungarian algorithm to solve this assignment problem efficiently, minimizing the total cost across the matrix. This allows it to determine the globally optimal set of assignments across frames, enabling robust trajectory reconstruction even under challenging conditions such as high particle density or noisy detections.

Examples and tutorials using LapTrack are available in the [LapTrack documentation](https://github.com/yfukai/laptrack/tree/main/docs/examples).

### Linking Localizations in Simulations

Apply the method to the localization dataframe.

In [ ]:
# Define the LapTrack instance to later link the detections.
laptrack = LapTrack(
    track_cost_cutoff=3**2, # Maximum allowed distance in pixels for linking detections across frames.
    gap_closing_cost_cutoff=5**2, # Max maximum allowed linking cost for missing detections.
    gap_closing_max_frame_count=5, # Maximum number of missing frames, or "memory".
    splitting_cutoff=False, # Disable cell division-like events.
)

# Fetch the predicted trajectories as the first output of the function.
sim_trajs_pred_method2, _, _ = laptrack.predict_dataframe(
    df=df_sim_video,  # DataFrame with detections
    coordinate_cols=["x", "y"],  # Name of columns containing coordinates
)

# Reset the indexing order to ensure sequential trajectory IDs.
sim_trajs_pred_method2 = sim_trajs_pred_method2.reset_index()

Create a trajectory list from the output dataframe.

In [ ]:
sim_trajs_pred_method2_list=[]

# Eliminate duplicates in track_id and create a list of trajectories.
for i in sim_trajs_pred_method2.track_id.drop_duplicates():
    traj = sim_trajs_pred_method2.loc[
        sim_trajs_pred_method2.track_id == i,
        ["frame", "x", "y"]
    ].values
    sim_trajs_pred_method2_list.append(traj)

# Filter trajectory lists shorter than 10 frames.
sim_trajs_pred_method2_list = [trajectory for trajectory in sim_trajs_pred_method2_list if len(trajectory) >= 15]
print(f"Number of trajectories found: {len(sim_trajs_pred_method2_list)}")

Create a video with overlayed localizations and trajectories.

In [ ]:
sim_video_method2_results = utils.make_video_with_trajs(
    trajs_pred_list=sim_trajs_pred_method2_list,
    video=sim_video,
    fov_size=sim_image_size,
    trajs_gt_list=sim_trajs_gt_list,
)

# Display the video.
sim_video_method2_results

### Evaluating Linking Performance

Evaluate the overall performance of the tracking.

In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_gt_list,
    sim_trajs_pred_method2_list,
    eps=5,
);

Display the reconstructed trajectories together with the ground truth.

In [ ]:
# Compute the total squared distance between all trajectories to match
# predicted trajectories with ground truth.
matched_pairs, _, _ = utils.trajectory_assignment(
    sim_trajs_gt_list,
    sim_trajs_pred_method2_list,
    eps=5,
)

# Plot the trajectories.
utils.plot_trajectory_matches(
    sim_trajs_gt_list, sim_trajs_pred_method2_list, matched_pairs,
)

Calculate the time-averaged MSD for all the trajectories and compare curves obtained for matching trajectories (same color).

In [ ]:
utils.plot_TAMSDs(
    trajs_pred=sim_trajs_pred_method2_list,
    trajs_gt=sim_trajs_gt_list,
    matched_pairs=matched_pairs,
) 

### Linking Localizations in Experiments

Apply the same steps to track the experiment and visualize the results.

In [ ]:
# Link detections across frames into trajectories using laptrack.
exp_trajs_pred_method2, _, _ = laptrack.predict_dataframe(
    df_exp_video,
    ["x", "y"],
    only_coordinate_cols=True,
)

exp_trajs_pred_method2 = exp_trajs_pred_method2.reset_index()

# Create a list to store trajectories.
exp_trajs_pred_method2_list=[]
for i in exp_trajs_pred_method2.track_id.drop_duplicates():
    traj = exp_trajs_pred_method2.loc[
        exp_trajs_pred_method2.track_id == i,
        ["frame", "x", "y"]
    ].values
    exp_trajs_pred_method2_list.append(traj)

# Filter trajectory lists shorter than 10 frames.
exp_trajs_pred_method2_list = [trajectory for trajectory in exp_trajs_pred_method2_list if len(trajectory) >= 15]
print(f"Number of trajectories found: {len(exp_trajs_pred_method2_list)}")

# Visualize the video with the trajectories overlaid.
exp_video_method2_results = utils.make_video_with_trajs(
    trajs_pred_list=exp_trajs_pred_method2_list,
    video=exp_video,
    fov_size=exp_image_size,
)

# Display the video.
exp_video_method2_results

Calculate the time-averaged MSD for the trajectories.

In [ ]:
utils.plot_TAMSDs(trajs_pred=exp_trajs_pred_method2_list)

## Method 3: MAGIK

MAGIK is a tracking framework designed to analyze the motion of dynamic systems, including cells, bacteria, individual molecules, colloids, and other active particles. The name stands for Motion Analysis through Graph Neural Network Inductive Knowledge.

At its core, MAGIK uses graph neural networks (GNNs) to learn patterns in particle movement and to infer trajectories across frames. This data-driven approach enables MAGIK to outperform traditional tracking methods in challenging conditions, such as dense particle fields, complex interaction dynamics, or non-Brownian motion.

Thanks to its ability to learn and generalize motion priors, MAGIK is particularly effective in noisy or ambiguous experimental settings, making it a strong complement—or even an alternative—to classical tools like TrackPy and LapTrack.

For more details, see the publication  
[Geometric Deep Learning Reveals the Spatiotemporal Features of Microscopic Motion](https://www.nature.com/articles/s42256-022-00595-0) *Nat Mach Intell* **5**, 71–82 (2023).

### Training MAGIK with Simulations

To train MAGIK effectively, you first need to generate appropriate training data. Since the experimental dataset in this case features colloids undergoing Brownian motion, you'll use the `simulate_Brownian_trajs()` function to produce groups of synthetic trajectories that replicate this behavior.

**Note:** It is crucial that the simulated training data accurately reflect the motion characteristics of your experimental particles. If your system exhibits pure diffusion (Brownian motion), the training simulations should mirror that. Conversely, if your experimental data involve additional dynamics—such as drift, confinement, or driven flow (e.g. in nanofluidic systems)—these should be incorporated into the training data to ensure MAGIK learns the correct motion priors.

In [ ]:
# Parameters for training dataset.
train_dataset_size = 100  # Number of videos
train_number_particles = 20  # Number of particles per video
train_number_timesteps = 50  # Number of frames per video
train_fov_size = 128  # Size of the field of view

# Initiate a dataframe containing all the simulated trajectories.
df_train_dataset = []
for video_index in range(train_dataset_size):
    # Simulate trajectories for one video.
    sim_trajs_train_dataset = utils.simulate_Brownian_trajs(
        num_particles=train_number_particles,
        num_timesteps=train_number_timesteps,
        fov_size=train_fov_size,
        diffusion_std=np.random.uniform(0.1, 2.0),
    )
    # Break trajectories going in/out of FOV.
    sim_trajs_train_dataset_list = utils.traj_break(
        trajs=sim_trajs_train_dataset,
        fov_size=train_fov_size,
        num_particles=train_number_particles,
    )
    # Make into dataframe with "frame" (which frame in the video),
    # label(which particle in that frame), set (which video).
    for traj_index, traj in enumerate(sim_trajs_train_dataset_list):
        df_traj = pd.DataFrame(
            traj[:, 1:],
            columns=["centroid-0", "centroid-1"],
        )
        df_traj["frame"] = traj[:, 0].astype(int)
        df_traj["label"] = traj_index
        df_traj["set"] = f"{video_index}"
        df_train_dataset.append(df_traj)

# Concatenate to dataframe.
df_train_dataset = pd.concat(df_train_dataset, ignore_index=True)

Normalize the trajectory coordinates between 0 and 1 by dividing for the fov size.

In [ ]:
# Normalize centroids between 0 and 1.
norm_factor = np.array([train_fov_size, train_fov_size])
df_train_dataset.loc[:, df_train_dataset.columns.str.contains("centroid")] = (
    df_train_dataset.loc[
        :,
        df_train_dataset.columns.str.contains("centroid")
    ] / norm_factor
)

To train MAGIK with the simulated trajectories, you need to produce a graph representation with the function `GraphFromTrajectories()`, defined in the utility file `utils.py`.

In [ ]:
# Instance the graph constructor.
graph_constructor = utils.GraphFromTrajectories(
    connectivity_radius=0.01,
    max_frame_distance=3,
)

# Generate graph from training data using graph constructor.
train_dataset_graph = graph_constructor(df=df_train_dataset)

print(train_dataset_graph)

Define the augmentation pipeline and the dataloader.

In [ ]:
# Initialize the graph dataset class.
# `Dt` is the time difference between frames to sample from the graph.
# Specify augmentations with transform,
# NodeDropout() should be last.
train_dataset = utils.GraphDataset(
    train_dataset_graph,
    dataset_size=train_dataset_size,
    Dt=5,
    transform=Compose(
        [
            utils.RandomRotation(),
            utils.RandomFlip(),
            utils.NodeDropout(),
        ]
    )
)

# Initialize the training data loader.
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
)

Define the hyperparameters of the architecture.

In [ ]:
magik = dl.GraphToEdgeMAGIK(
    [96,] * 4,1,
    out_activation=torch.nn.Sigmoid,
)

magik.encoder[0].configure(
    hidden_features=[32, 64],
    out_features=96,
    out_activation=torch.nn.ReLU,
)

magik.encoder[1].configure(
    hidden_features=[32, 64],
    out_features=96,
    out_activation=torch.nn.ReLU,
)

magik.head.configure(hidden_features=[64, 32]);

classifier_magik = dl.BinaryClassifier(
    model=magik,
    optimizer=dl.Adam(lr=1e-3),
).build()


# Set training parameters and train.
trainer_magik = dl.Trainer(max_epochs=64, accelerator="auto")

Train the model or load preexisting weights.

In [ ]:
# Define whether to force training even if weights exist.
force_training = False

# Define folder and model path.
folder_name = "MAGIK"
magik_model_path = os.path.join(folder_name, "magik_weights.pth")

# Train or load weights.
if not os.path.isfile(magik_model_path) or force_training:
    print("Training MAGIK model (either forced or weights not found).")

    # Ensure save directory exists.
    os.makedirs(folder_name, exist_ok=True)

    # Train the model.
    trainer_magik.fit(classifier_magik, train_loader)

    # Plot training history.
    trainer_magik.history.plot()
    
    # Save trained weights.
    torch.save(magik.state_dict(), magik_model_path)
    print(f"Saved MAGIK weights to '{magik_model_path}'.")
else:
    # Load pre-trained weights.
    magik.load_state_dict(torch.load(magik_model_path, weights_only=True))
    print(f"Loaded preexisting MAGIK weights from '{magik_model_path}'.")

# Set the model to evaluation mode.
classifier_magik.eval();

# Transfer the model to the best available device (optional).
classifier_magik.to(device);

### Linking Localizations in Simulations

You'll use the trajectories in the simulated video as the test dataset for the trained model of MAGIK. First, format the simulated dataframe. Then, convert the localizations corresponding to the simulated trajectories into a graph.

In [ ]:
# Rename for compatibility with label format.
df_sim_video_formatted = df_sim_video.rename(
    columns={"x": "centroid-0", "y": "centroid-1"}
)

# Add label, set, and solution columns.
df_sim_video_formatted[["label", "set", "solution"]] = 0

# Normalize coordinates to [0, 1].
frame_height, frame_width, _ = sim_image.shape
df_sim_video_formatted[["centroid-0", "centroid-1"]] /= [frame_width,
                                                         frame_height]

# Generate a graph from graph_constructor. As test_graph returns a list of
#  graphs, we select the first element from the list as it only has 1 element.
sim_video_graph = graph_constructor(df=df_sim_video_formatted)[0]

# Transfer graph to the best available device (optional).
sim_video_graph = sim_video_graph.to(device)

Apply the trained model of MAGIK to predict the edge features.

In [ ]:
# Perform prediction on test graph.
sim_trajs_edges_pred_method3 = classifier_magik(sim_video_graph)

# Apply threshold to get binary edge predictions.
sim_trajs_edges_pred_method3 = \
    sim_trajs_edges_pred_method3.cpu().detach().numpy() > 0.5

Get the ground-truth edge features and, as a first performance metrics, use the F1-score for the classification of the edges.

In [ ]:
# Get the ground truth edges from the graph.
sim_trajs_edges_gt = sim_video_graph.y.cpu()  # Transfer to CPU

# Compute the F1 score.
F1 = f1_score(sim_trajs_edges_gt, sim_trajs_edges_pred_method3)
print(f"Test F1 score: {F1}")

The edge feature can be used to obtain the predicted trajectories using the dedicate class.

In [ ]:
# Compute the trajectories from the predicted edges.
trajectory_constructor = utils.ComputeTrajectories()
sim_trajs_pred_method3 = trajectory_constructor(
    sim_video_graph.cpu(),
    sim_trajs_edges_pred_method3.squeeze(),
)

# Convert the predicted trajectories to a list format.
sim_trajs_pred_method3_list = utils.make_list(
    sim_trajs_pred_method3, sim_video_graph, sim_image_size,
)

# Filter trajectory lists shorter than 10 frames.
sim_trajs_pred_method3_list = [trajectory for trajectory in sim_trajs_pred_method3_list if len(trajectory) >= 15]
print(f"Number of trajectories found: {len(sim_trajs_pred_method3_list)}")


Create a video with overlayed localizations and trajectories.

In [ ]:
# Create a video with the predicted trajectories.
sim_video_method3_results = utils.make_video_with_trajs(
    trajs_pred_list=sim_trajs_pred_method3_list,
    video=sim_video,
    fov_size=sim_image_size,
    trajs_gt_list=sim_trajs_gt_list,
)

# Plot the video.
sim_video_method3_results

### Evaluating Linking Performance

Evaluate the overall performance of the tracking.

In [ ]:
# Evaluate performance metrics.
utils.trajectory_metrics(
    sim_trajs_gt_list,
    sim_trajs_pred_method3_list,
    eps=5,
);

Display the reconstructed trajectories together with the groud truth.

In [ ]:
#  Compute the total squared distance between all trajectories to match
#  predicted trajectories with ground truth.
matched_pairs, _, _ = utils.trajectory_assignment(
    sim_trajs_gt_list,
    sim_trajs_pred_method3_list,
    eps=5,
)

# Plot the trajectories.
utils.plot_trajectory_matches(
    sim_trajs_gt_list, sim_trajs_pred_method3_list, matched_pairs,
)

Calculate the time-averaged MSD for all the trajectories and compare curves obtained for matching trajectories (same color).

In [ ]:
utils.plot_TAMSDs(
    trajs_pred = sim_trajs_pred_method3_list,
    trajs_gt = sim_trajs_gt_list,
    matched_pairs = matched_pairs,
) 

### Linking Localizations in Experiments

Apply the same steps to track the experiment and visualize the results.

In [ ]:
# Rename for compatibility with label format.
df_exp_video_formatted = df_exp_video.rename(
    columns={"x": "centroid-0", "y": "centroid-1"}
)

# Add label, set, and solution columns.
df_exp_video_formatted[["label", "set", "solution"]] = 0

# Normalize coordinates to [0, 1].
frame_height, frame_width, _ = sim_image.shape
df_exp_video_formatted[["centroid-0", "centroid-1"]] /= [frame_width,
                                                         frame_height]

# Generate a graph from graph_constructor. As test_graph returns a list of
#  graphs, we select the first element from the list as it only has 1 element.
exp_video_graph = graph_constructor(df=df_exp_video_formatted)[0].to(device)

# Perform prediction on graph.
exp_trajs_edges_pred_method3 = classifier_magik(exp_video_graph)
exp_trajs_edges_pred_method3 = \
    exp_trajs_edges_pred_method3.cpu().detach().numpy() > 0.5

# Compute the trajectories from the predicted edges.
trajectory_constructor = utils.ComputeTrajectories()
sim_trajs_pred_method3 = trajectory_constructor(
    exp_video_graph.cpu(),
    exp_trajs_edges_pred_method3.squeeze(),
)

# Convert the predicted trajectories to a list format.
exp_trajs_pred_method3_list = utils.make_list(
    sim_trajs_pred_method3, exp_video_graph, exp_image_size,
)

# Filter trajectory lists shorter than 10 frames.
exp_trajs_pred_method3_list = [trajectory for trajectory in exp_trajs_pred_method3_list if len(trajectory) >= 15]
print(f"Number of trajectories found: {len(exp_trajs_pred_method3_list)}")

# Create a video with the predicted trajectories.
exp_video_method3_results = utils.make_video_with_trajs(
    trajs_pred_list=exp_trajs_pred_method3_list,
    video=exp_video,
    fov_size=exp_image_size,
)

exp_video_method3_results

Calculate the time-averaged MSD for the trajectories.

In [ ]:
utils.plot_TAMSDs(trajs_pred=exp_trajs_pred_method3_list)